In [1]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader

print("Python:", sys.version)
print("PyTorch:", torch.__version__)

Python: 3.11.15 (main, Aug  2 2026, 19:10:42) [GCC 11.4.0]
PyTorch: 2.1.2+cu118


In [2]:
# Root del progetto
PROJECT_ROOT = Path("..").resolve()

# Root del dataset ATLAS-120k
DATASET_ROOT = Path("/work/tesi_cmininno/dataset/atlas120k").resolve()

print("Project root :", PROJECT_ROOT)
print("Dataset root :", DATASET_ROOT)

Project root : /homes/cmininno/tesi/surgical_scene_segmentation
Dataset root : /work/tesi_cmininno/dataset/atlas120k


In [ ]:
assert PROJECT_ROOT.exists(), (
    f"Project root non trovato: {PROJECT_ROOT}"
)

assert DATASET_ROOT.exists(), (
    f"ATLAS-120k non trovato: {DATASET_ROOT}"
)

print("Project root: OK")
print("Dataset root: OK")

In [ ]:
splits = ["train", "val", "test"]

for split in splits:
    split_path = DATASET_ROOT / split

    print(
        f"{split:>5}: "
        f"{'OK' if split_path.exists() else 'MISSING'} "
        f"-> {split_path}"
    )

In [3]:
sys.path.insert(0, str(PROJECT_ROOT))

from src.datasets.base_dataset import BaseSegmentationDataset
from src.datasets.atlas120k_dataset import ATLAS120KDataset

print("BaseSegmentationDataset:", BaseSegmentationDataset)
print("ATLAS120KDataset:", ATLAS120KDataset)

BaseSegmentationDataset: <class 'src.datasets.base_dataset.BaseSegmentationDataset'>
ATLAS120KDataset: <class 'src.datasets.atlas120k_dataset.ATLAS120KDataset'>


In [ ]:
print("Numero classi:", ATLAS120KDataset.NUM_CLASSES)
print()

for idx, name in enumerate(ATLAS120KDataset.CLASS_NAMES):
    print(f"{idx:2d} -> {name}")

In [ ]:
assert ATLAS120KDataset.NUM_CLASSES == 30
assert len(ATLAS120KDataset.CLASS_NAMES) == 30
assert ATLAS120KDataset.NUM_ORIGINAL_CLASSES == 47

assert len(ATLAS120KDataset.ORIGINAL_TO_TRAIN_ID) == 47

print("Taxonomy check: OK")

In [ ]:
assert ATLAS120KDataset.NUM_CLASSES == 30
assert len(ATLAS120KDataset.CLASS_NAMES) == 30
assert ATLAS120KDataset.NUM_ORIGINAL_CLASSES == 47

assert len(ATLAS120KDataset.ORIGINAL_TO_TRAIN_ID) == 47

print("Taxonomy check: OK")

In [4]:
mapping = ATLAS120KDataset.ORIGINAL_TO_TRAIN_ID

print("Original ID -> Training ID")
print()

for original_id, train_id in enumerate(mapping):
    print(
        f"{original_id:2d} -> {train_id:2d} "
        f"({ATLAS120KDataset.CLASS_NAMES[train_id]})"
    )

Original ID -> Training ID

 0 ->  0 (Background)
 1 ->  1 (Tools/camera)
 2 ->  2 (Vein)
 3 ->  3 (Artery)
 4 ->  4 (Nerve)
 5 ->  5 (Small intestine)
 6 ->  6 (Colon/rectum)
 7 ->  7 (Abdominal wall)
 8 ->  8 (Diaphragm)
 9 ->  9 (Fat)
10 ->  3 (Artery)
11 ->  2 (Vein)
12 -> 10 (Liver)
13 -> 11 (Bile/lymph duct)
14 -> 12 (Gallbladder)
15 ->  2 (Vein)
16 -> 13 (Hepatic ligament)
17 -> 14 (Cystic plate)
18 -> 15 (Stomach)
19 -> 11 (Bile/lymph duct)
20 ->  9 (Fat)
21 -> 11 (Bile/lymph duct)
22 -> 16 (Spleen)
23 -> 17 (Uterus)
24 -> 18 (Ovary)
25 -> 19 (Oviduct)
26 -> 20 (Prostate)
27 -> 21 (Urethra)
28 -> 22 (Ligated plexus)
29 -> 23 (Seminal vesicles)
30 -> 24 (Non anatomical)
31 -> 25 (Bladder)
32 ->  0 (Background)
33 -> 26 (Lung)
34 -> 27 (Airway (bronchus/trachea))
35 -> 28 (Esophagus)
36 -> 29 (Pericardium)
37 ->  2 (Vein)
38 -> 11 (Bile/lymph duct)
39 ->  4 (Nerve)
40 ->  0 (Background)
41 -> 24 (Non anatomical)
42 ->  0 (Background)
43 ->  0 (Background)
44 ->  0 (Background)
45

In [7]:
train_dataset = ATLAS120KDataset(
    root=DATASET_ROOT / "train",
    validate_clip_index=False,
)

print("Train samples:", len(train_dataset))

FileNotFoundError: Missing images directory:
/work/tesi_cmininno/dataset/atlas120k/train/adrenalectomy/16GPCUPkXYQ/clip_0001/images

In [6]:
val_dataset = ATLAS120KDataset(
    root=DATASET_ROOT / "val",
    validate_clip_index=True,
)


FileNotFoundError: Missing images directory:
/work/tesi_cmininno/dataset/atlas120k/val/cholecystectomy/1trsPYQ7rmg/clip_0001/images

In [ ]:

test_dataset = ATLAS120KDataset(
    root=DATASET_ROOT / "test",
    validate_clip_index=True,
)

In [8]:
from pathlib import Path

root = Path("/work/tesi_cmininno/dataset/atlas120k/train")

only_masks = []
only_images = []
both = []
neither = []

for clip_dir in root.rglob("clip_*"):

    if not clip_dir.is_dir():
        continue

    has_images = (clip_dir / "images").exists()
    has_masks = (clip_dir / "masks").exists()

    if has_images and has_masks:
        both.append(clip_dir)
    elif has_masks:
        only_masks.append(clip_dir)
    elif has_images:
        only_images.append(clip_dir)
    else:
        neither.append(clip_dir)

print("Clip con images + masks:", len(both))
print("Clip solo masks:", len(only_masks))
print("Clip solo images:", len(only_images))
print("Clip senza entrambe:", len(neither))

print("\nEsempi solo masks:")
for p in only_masks[:10]:
    print(p)

Clip con images + masks: 5
Clip solo masks: 353
Clip solo images: 0
Clip senza entrambe: 0

Esempi solo masks:
/work/tesi_cmininno/dataset/atlas120k/train/sigmoidcolectomy/KA5RO0CMg6I/clip_0001
/work/tesi_cmininno/dataset/atlas120k/train/sigmoidcolectomy/MIFNpGu1cC8/clip_0001
/work/tesi_cmininno/dataset/atlas120k/train/sigmoidcolectomy/MIFNpGu1cC8/clip_0002
/work/tesi_cmininno/dataset/atlas120k/train/sigmoidcolectomy/2CmNyZWDxO4/clip_0001
/work/tesi_cmininno/dataset/atlas120k/train/sigmoidcolectomy/2CmNyZWDxO4/clip_0002
/work/tesi_cmininno/dataset/atlas120k/train/sigmoidcolectomy/2CmNyZWDxO4/clip_0003
/work/tesi_cmininno/dataset/atlas120k/train/sigmoidcolectomy/S3S_3GDc3dA/clip_0001
/work/tesi_cmininno/dataset/atlas120k/train/sigmoidcolectomy/S3S_3GDc3dA/clip_0002
/work/tesi_cmininno/dataset/atlas120k/train/gastric_surgery/4FHGGFZsPzw/clip_0004
/work/tesi_cmininno/dataset/atlas120k/train/gastric_surgery/4FHGGFZsPzw/clip_0001
